# SAM2 Video Segmentation
**Automatic Surgical Tool Segmentation in Video**

## 1. Installation
Run **only once** to install SAM2 and ffmpeg.

In [ ]:
!git clone https://github.com/facebookresearch/sam2.git
%cd sam2
!pip install -e .

# Install ffmpeg (Ubuntu):
# !sudo apt-get install ffmpeg -y

## 2. Download Checkpoint

In [ ]:
import urllib.request, os

ckpt_dir  = r"your_checkpoint_directory"  # Change this to your desired checkpoint directory
ckpt_path = os.path.join(ckpt_dir, "sam2_hiera_large.pt")
ckpt_url  = "https://dl.fbaipublicfiles.com/segment_anything_2/072824/sam2_hiera_large.pt"

if not os.path.exists(ckpt_path):
    print("Descargando checkpoint (~900 MB)...")
    urllib.request.urlretrieve(ckpt_url, ckpt_path)
    print("Descarga completada:", ckpt_path)
else:
    print("Checkpoint ya existe:", ckpt_path)

## 3. Imports

In [ ]:
import os

import cv2
import torch
import numpy as np
import matplotlib.pyplot as plt

from PIL import Image
from sam2.build_sam import build_sam2_video_predictor

## 4. Configuration

In [ ]:
VIDEO_PATH = r"path_to_your_video.mp4"  # change that to your video path

FRAMES_DIR = "frames"
MASKS_DIR  = "masks"

os.makedirs(FRAMES_DIR, exist_ok=True)
os.makedirs(MASKS_DIR,  exist_ok=True)

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
print("DEVICE:", DEVICE)

## 5. Extract Video Frames

In [ ]:
cap = cv2.VideoCapture(VIDEO_PATH)
frame_idx = 0

while True:
    ret, frame = cap.read()
    if not ret:
        break
    frame_path = os.path.join(FRAMES_DIR, f"{frame_idx:05d}.jpg")
    cv2.imwrite(frame_path, frame)
    frame_idx += 1

cap.release()
print("Extracted frames:", frame_idx)

## 6. Load SAM2

In [ ]:
checkpoint = r"your_checkpoint_directory/sam2_hiera_large.pt"  # Change this to your checkpoint path
model_cfg  = "configs/sam2/sam2_hiera_l.yaml"

predictor = build_sam2_video_predictor(
    model_cfg,
    checkpoint,
    device=DEVICE
)
print("SAM2 loaded")

## 7. Initialize Video State

In [ ]:
inference_state = predictor.init_state(video_path=FRAMES_DIR)
print("Inference state initialized")

## 8. Display First Frame

In [ ]:
frame_names = sorted(os.listdir(FRAMES_DIR))

first_frame = np.array(
    Image.open(os.path.join(FRAMES_DIR, frame_names[0]))
)

plt.figure(figsize=(10, 10))
plt.imshow(first_frame)
plt.title("First frame")
plt.axis("off")
plt.show()

## 9. Define Point on Surgical Tool



In [ ]:
# Show image to select coordenates
plt.figure(figsize=(14, 9))
plt.imshow(first_frame)
plt.title("Note X and Y")
# axes activated
plt.show()

In [ ]:
# *** CHANGE THESE VALUES ***
X = 1000
Y = 600
# ********************************

clicked_points = [[X, Y]]

# Previsualization 
plt.figure(figsize=(14, 9))
plt.imshow(first_frame)
plt.scatter([X], [Y], c='red', s=300, zorder=5)
plt.title(f"Point selected: ({X}, {Y})")
plt.axis("off")
plt.show()

In [ ]:
%matplotlib notebook

clicked_points = []

fig, ax = plt.subplots(figsize=(10, 10))
ax.imshow(first_frame)
ax.set_title("Click on the instrument to select it")
ax.axis("off")

scatter = ax.scatter([], [], c='red', s=200, zorder=5)

def on_click(event):
    if event.inaxes != ax:
        return
    x, y = int(event.xdata), int(event.ydata)
    clicked_points.clear()
    clicked_points.append([x, y])
    scatter.set_offsets([[x, y]])
    ax.set_title(f"Point selected: ({x}, {y})")
    fig.canvas.draw()
    print(f"Point saved: ({x}, {y})")

fig.canvas.mpl_connect('button_press_event', on_click)
plt.tight_layout()
plt.show()

## 10. Add Prompt

In [ ]:
ann_frame_idx = 0
ann_obj_id    = 1

if len(clicked_points) == 0:
    if 'X' in globals() and 'Y' in globals():
        clicked_points = [[X, Y]]
    else:
        raise RuntimeError(
            "X and Y are not defined. "
            "Define X and Y in previous cell or run the interactive cell."
        )

points = np.array(clicked_points, dtype=np.float32)
labels = np.array([1] * len(clicked_points), np.int32)

_, out_obj_ids, out_mask_logits = predictor.add_new_points(
    inference_state=inference_state,
    frame_idx=ann_frame_idx,
    obj_id=ann_obj_id,
    points=points,
    labels=labels,
)
print("Prompt added with point:", clicked_points)

## 11. Display Initial Mask

In [ ]:
mask = (out_mask_logits[0] > 0.0).cpu().numpy()
if mask.ndim == 3 and mask.shape[0] == 1:
    mask = mask[0]

plt.figure(figsize=(10, 10))
plt.imshow(first_frame)
plt.imshow(mask, alpha=0.5)
plt.title("Initial mask")
plt.axis("off")
plt.show()

## 12. Propagate Through Video

In [ ]:
video_segments = {}

for out_frame_idx, out_obj_ids, out_mask_logits in predictor.propagate_in_video(inference_state):
    video_segments[out_frame_idx] = {
        out_obj_id: (out_mask_logits[i] > 0.0).cpu().numpy()
        for i, out_obj_id in enumerate(out_obj_ids)
    }

print("Segmentation completed")

## 13. Save Masks

In [ ]:
for frame_idx, segments in video_segments.items():
    for obj_id, mask in segments.items():
        mask_uint8 = (mask * 255).astype(np.uint8)
        save_path  = os.path.join(MASKS_DIR, f"{frame_idx:05d}.png")
        cv2.imwrite(save_path, mask_uint8)

print("Masks saved")

## 14. Visualize Results
Change `VIS_FRAME` to inspect any frame.

In [ ]:
VIS_FRAME = 50

img  = np.array(Image.open(os.path.join(FRAMES_DIR, frame_names[VIS_FRAME])))
mask = video_segments[VIS_FRAME][1]
if mask.ndim == 3 and mask.shape[0] == 1:
    mask = mask[0]

plt.figure(figsize=(10, 10))
plt.imshow(img)
plt.imshow(mask, alpha=0.5)
plt.title(f"Frame {VIS_FRAME}")
plt.axis("off")
plt.show()

## 15. Export Video with Overlay

In [ ]:
output_video = "segmented_video.mp4"
h, w = img.shape[:2]

writer = cv2.VideoWriter(
    output_video,
    cv2.VideoWriter_fourcc(*'mp4v'),
    30,
    (w, h)
)

for idx in range(len(frame_names)):
    frame = cv2.imread(os.path.join(FRAMES_DIR, frame_names[idx]))
    frame = cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)

    if idx in video_segments:
        mask = video_segments[idx][1]
        if mask.ndim == 3 and mask.shape[0] == 1:
            mask = mask[0]
        overlay = frame.copy()
        overlay[mask] = [255, 0, 0]
        frame   = cv2.addWeighted(overlay, 0.5, frame, 0.5, 0)

    writer.write(cv2.cvtColor(frame, cv2.COLOR_RGB2BGR))

writer.release()
print("Video exported:", output_video)